# BanglaBERT six-class comparison

This notebook adds one reproducible, imbalance-aware experiment using the official
`csebuetnlp/banglabert` checkpoint. It preserves the official BanglaMultiHate
train/dev/test partitions and the exact six-class mapping used by the corrected
classical experiment.

**Paper-valid outputs are written only when `SMOKE_TEST = False`.** The default smoke
mode validates data, normalization, tokenization, weighted loss, model forward pass,
and a tiny training path without evaluating the test split. For the final experiment,
open this notebook in Google Colab with a GPU, set `SMOKE_TEST = False`, and run all cells.

In [ ]:
# Install reproducible notebook dependencies without replacing Colab's CUDA-enabled PyTorch.
%pip install -q -U "transformers>=4.48,<6" "datasets>=3,<6" "accelerate>=1,<2"             "scikit-learn>=1.4" "pandas>=2.2" "numpy>=1.26" "matplotlib>=3.8" "seaborn>=0.13"
%pip install -q "git+https://github.com/csebuetnlp/normalizer.git"

In [ ]:
# Imports, fixed experiment configuration, repository discovery, and deterministic setup.
import inspect
import json
import math
import os
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "600")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
import platform
import random
import re
import time
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torch.nn.functional as F
import transformers
from datasets import Dataset, DatasetDict
from IPython.display import Markdown, display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_CHECKPOINT = "csebuetnlp/banglabert"
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 1

# Keep True for local validation. Change to False only in a CUDA-enabled Colab runtime
# to produce the paper-valid test results and comparison artifacts.
SMOKE_TEST = True
SMOKE_TRAIN_PER_CLASS = 8
SMOKE_DEV_PER_CLASS = 4
SMOKE_MAX_STEPS = 2

CLASS_ORDER = ["None", "Abusive", "Sexism", "Religious Hate", "Political Hate", "Profane"]
LABEL_TO_ID = {label: index for index, label in enumerate(CLASS_ORDER)}
ID_TO_LABEL = {index: label for label, index in LABEL_TO_ID.items()}
MINORITY_LABELS = ["Sexism", "Religious Hate", "Profane"]
MINORITY_IDS = [LABEL_TO_ID[label] for label in MINORITY_LABELS]
EXPECTED_ROWS = {"train": 35522, "dev": 5024, "test": 10200}
EXPECTED_NONE_COUNTS = {"train": 19954, "dev": 2898, "test": 5751}

def locate_project_root():
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "processed_data" / "train_clean.csv").exists()
            and (candidate / "phase4_outputs" / "phase4_manifest.json").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Repository root not found. In Colab, clone or mount this repository and run the notebook from it."
    )

PROJECT_ROOT = locate_project_root()
OUTPUT_DIR = PROJECT_ROOT / "phase5_outputs" / "banglabert"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Phase 5 output directory:", OUTPUT_DIR)
print("SMOKE_TEST:", SMOKE_TEST)

In [ ]:
# Hardware gate: full training is permitted only on CUDA; smoke validation may use CPU.
CUDA_AVAILABLE = bool(torch.cuda.is_available())
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None
GPU_FREE_GB = None
GPU_TOTAL_GB = None
CUDA_CAPABILITY = None
if CUDA_AVAILABLE:
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    GPU_FREE_GB = free_bytes / (1024 ** 3)
    GPU_TOTAL_GB = total_bytes / (1024 ** 3)
    CUDA_CAPABILITY = torch.cuda.get_device_capability(0)

hardware = {
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "cuda_available": CUDA_AVAILABLE,
    "gpu_name": GPU_NAME,
    "gpu_free_memory_gb": GPU_FREE_GB,
    "gpu_total_memory_gb": GPU_TOTAL_GB,
    "cuda_capability": CUDA_CAPABILITY,
}
print(json.dumps(hardware, indent=2))

if not CUDA_AVAILABLE:
    display(Markdown(
        "**CUDA is unavailable. Full training is intentionally blocked on this machine. "
        "Run this notebook in Google Colab with a GPU and set `SMOKE_TEST = False` for final results.**"
    ))
    assert SMOKE_TEST, "Full BanglaBERT training on CPU is prohibited; enable a Colab GPU."

In [ ]:
# Load the corrected official split tables without interpreting the legitimate label "None" as missing.
split_frames = {}
for split_name in ["train", "dev", "test"]:
    csv_path = PROJECT_ROOT / "processed_data" / f"{split_name}_clean.csv"
    split_df = pd.read_csv(csv_path, keep_default_na=False)
    required_columns = {"id", "text_raw", "type_of_hate", "type_of_hate_id", "split"}
    missing_columns = sorted(required_columns - set(split_df.columns))
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {missing_columns}")
    split_frames[split_name] = split_df

split_validation_records = []
class_distribution_records = []
for split_name, split_df in split_frames.items():
    labels = split_df["type_of_hate"].astype(str)
    observed_classes = set(labels.unique())
    mapped_ids = labels.map(LABEL_TO_ID)

    assert len(split_df) == EXPECTED_ROWS[split_name], (
        f"{split_name} row count changed: {len(split_df)} != {EXPECTED_ROWS[split_name]}"
    )
    assert observed_classes == set(CLASS_ORDER), (
        f"{split_name} does not contain exactly the fixed six classes: {sorted(observed_classes)}"
    )
    assert labels.nunique() == 6
    assert int(labels.eq("None").sum()) == EXPECTED_NONE_COUNTS[split_name]
    assert mapped_ids.notna().all()
    assert np.array_equal(mapped_ids.to_numpy(dtype=int), split_df["type_of_hate_id"].to_numpy(dtype=int))
    assert split_df["split"].astype(str).eq(split_name).all()
    assert split_df["text_raw"].astype(str).str.strip().ne("").all()

    split_validation_records.append({
        "split": split_name,
        "rows": int(len(split_df)),
        "target_classes": int(labels.nunique()),
        "none_count": int(labels.eq("None").sum()),
    })
    counts = labels.value_counts().reindex(CLASS_ORDER, fill_value=0)
    for label in CLASS_ORDER:
        class_distribution_records.append({
            "split": split_name,
            "class": label,
            "class_id": LABEL_TO_ID[label],
            "count": int(counts[label]),
            "percentage": float(100 * counts[label] / len(split_df)),
        })

split_validation_df = pd.DataFrame(split_validation_records)
class_distribution_df = pd.DataFrame(class_distribution_records)
display(split_validation_df)
display(class_distribution_df)
print("All exact split-size, six-class, and legitimate None-label assertions passed.")

In [ ]:
# Load and verify the corrected classical reference directly from its machine-readable manifest.
phase4_manifest_path = PROJECT_ROOT / "phase4_outputs" / "phase4_manifest.json"
with open(phase4_manifest_path, "r", encoding="utf-8") as file:
    phase4_manifest = json.load(file)

expected_classical = {
    "accuracy": 0.6736274509803921,
    "micro_f1": 0.6736274509803921,
    "macro_f1": 0.5129250157090423,
    "weighted_f1": 0.6809862217986998,
    "balanced_accuracy": 0.5441650451497517,
}
assert phase4_manifest["selected_model"] == "Logistic Regression OVR - Balanced"
assert phase4_manifest["selected_feature_set"] == "text_numeric"
assert phase4_manifest["observed_class_labels"] == CLASS_ORDER
for metric_name, expected_value in expected_classical.items():
    actual_value = float(phase4_manifest["test_metrics"][metric_name])
    assert math.isclose(actual_value, expected_value, rel_tol=0.0, abs_tol=1e-12), (
        f"Classical {metric_name} mismatch: {actual_value} != {expected_value}"
    )

classical_reference = {
    "model": phase4_manifest["selected_model"],
    "feature_set": phase4_manifest["selected_feature_set"],
    "development_macro_f1": float(phase4_manifest["development_metrics"]["macro_f1"]),
    **{f"test_{name}": float(value) for name, value in expected_classical.items()},
    "minority_recall_mean": float(phase4_manifest["test_metrics"]["minority_recall_mean"]),
}
display(pd.DataFrame([classical_reference]))

In [ ]:
# Compute imbalance weights exclusively from the full official training labels and save them.
full_train_labels = split_frames["train"]["type_of_hate"].map(LABEL_TO_ID).to_numpy(dtype=np.int64)
train_class_counts = np.bincount(full_train_labels, minlength=len(CLASS_ORDER))
class_weights_np = len(full_train_labels) / (len(CLASS_ORDER) * train_class_counts.astype(np.float64))
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32)
class_weight_df = pd.DataFrame({
    "class_id": range(len(CLASS_ORDER)),
    "class": CLASS_ORDER,
    "training_count": train_class_counts,
    "weighted_cross_entropy_weight": class_weights_np,
})
display(class_weight_df)

class_weight_payload = {
    "source_split": "train",
    "formula": "n_samples / (n_classes * class_count)",
    "training_rows": int(len(full_train_labels)),
    "weights": {label: float(class_weights_np[index]) for index, label in enumerate(CLASS_ORDER)},
}
with open(OUTPUT_DIR / "class_weights.json", "w", encoding="utf-8") as file:
    json.dump(class_weight_payload, file, ensure_ascii=False, indent=2)
print("Saved training-only class weights to", OUTPUT_DIR / "class_weights.json")

In [ ]:
# Apply the official csebuetnlp Bangla normalization pipeline before tokenization.
try:
    from normalizer import normalize as official_bangla_normalize
except Exception as normalizer_error:
    raise RuntimeError(
        "The official csebuetnlp/normalizer package is required for BanglaBERT. Rerun the install cell."
    ) from normalizer_error

for split_name, split_df in split_frames.items():
    normalized = split_df["text_raw"].astype(str).map(official_bangla_normalize)
    assert normalized.str.strip().ne("").all(), f"Official normalization emptied text in {split_name}."
    split_frames[split_name] = split_df.assign(model_text=normalized)

print("Official Bangla normalization completed for train, dev, and test.")

In [ ]:
# Select small stratified train/dev subsets only in smoke mode; never evaluate test in smoke mode.
def stratified_sample(dataframe, rows_per_class):
    sampled_groups = []
    for label in CLASS_ORDER:
        class_rows = dataframe.loc[dataframe["type_of_hate"].eq(label)]
        sampled_groups.append(
            class_rows.sample(n=min(rows_per_class, len(class_rows)), random_state=SEED)
        )
    return pd.concat(sampled_groups, ignore_index=True)

if SMOKE_TEST:
    train_frame = stratified_sample(split_frames["train"], SMOKE_TRAIN_PER_CLASS)
    dev_frame = stratified_sample(split_frames["dev"], SMOKE_DEV_PER_CLASS)
    test_frame = None
    display(Markdown(
        "**SMOKE TEST ONLY — these sampled results are invalid for the paper, and the test split is not evaluated.**"
    ))
else:
    train_frame = split_frames["train"].copy()
    dev_frame = split_frames["dev"].copy()
    test_frame = split_frames["test"].copy()

for frame in [train_frame, dev_frame] + ([] if test_frame is None else [test_frame]):
    frame["labels"] = frame["type_of_hate"].map(LABEL_TO_ID).astype(int)

print("Training rows used:", len(train_frame))
print("Development rows used:", len(dev_frame))
print("Test evaluation enabled:", test_frame is not None)

In [ ]:
# Load the official checkpoint and verify tokenizer/model compatibility on one small training batch.
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(CLASS_ORDER),
    label2id=LABEL_TO_ID,
    id2label=ID_TO_LABEL,
)

probe_texts = train_frame["model_text"].iloc[:2].tolist()
probe_batch = tokenizer(
    probe_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)
probe_device = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
model.to(probe_device)
with torch.no_grad():
    probe_logits = model(**{name: tensor.to(probe_device) for name, tensor in probe_batch.items()}).logits
assert tuple(probe_logits.shape) == (2, len(CLASS_ORDER))
assert torch.isfinite(probe_logits).all()
model.to("cpu")
if CUDA_AVAILABLE:
    torch.cuda.empty_cache()

PARAMETER_COUNT = int(sum(parameter.numel() for parameter in model.parameters()))
TRAINABLE_PARAMETER_COUNT = int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))
print("Tokenizer vocabulary size:", len(tokenizer))
print("Small-batch logits shape:", tuple(probe_logits.shape))
print("Total parameters:", f"{PARAMETER_COUNT:,}")
print("Trainable parameters:", f"{TRAINABLE_PARAMETER_COUNT:,}")

In [ ]:
# Convert the selected frames to Hugging Face datasets and tokenize raw normalized text.
dataset_frames = {"train": train_frame, "dev": dev_frame}
if test_frame is not None:
    dataset_frames["test"] = test_frame

raw_dataset = DatasetDict({
    split_name: Dataset.from_pandas(
        frame[["id", "model_text", "labels"]], preserve_index=False
    )
    for split_name, frame in dataset_frames.items()
})

def tokenize_batch(batch):
    return tokenizer(batch["model_text"], truncation=True, max_length=MAX_LENGTH)

tokenized_dataset = raw_dataset.map(tokenize_batch, batched=True, desc="Tokenizing")
tokenized_dataset = tokenized_dataset.remove_columns(["id", "model_text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

for split_name, split_dataset in tokenized_dataset.items():
    assert len(split_dataset) == len(dataset_frames[split_name])
    assert set(split_dataset["labels"]) == set(range(len(CLASS_ORDER)))
    print(split_name, len(split_dataset), split_dataset.column_names)

In [ ]:
# Define consistent aggregate, per-class, probability, and confusion-matrix evaluation.
def stable_softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exponentiated = np.exp(shifted)
    return exponentiated / exponentiated.sum(axis=1, keepdims=True)

def metric_bundle(y_true, logits):
    logits = logits[0] if isinstance(logits, tuple) else np.asarray(logits)
    probabilities = stable_softmax(logits)
    predictions = probabilities.argmax(axis=1)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        predictions,
        labels=list(range(len(CLASS_ORDER))),
        zero_division=0,
    )
    aggregate = {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "micro_f1": float(f1_score(y_true, predictions, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, predictions, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, predictions, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, predictions, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, predictions, average="macro", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, predictions)),
        "cohen_kappa": float(cohen_kappa_score(y_true, predictions)),
        "minority_recall_mean": float(np.mean(recall[MINORITY_IDS])),
    }
    one_hot_true = np.eye(len(CLASS_ORDER), dtype=np.int64)[np.asarray(y_true, dtype=int)]
    try:
        aggregate["roc_auc_macro_ovr"] = float(
            roc_auc_score(one_hot_true, probabilities, average="macro", multi_class="ovr")
        )
    except ValueError:
        aggregate["roc_auc_macro_ovr"] = None
    try:
        aggregate["average_precision_macro"] = float(
            average_precision_score(one_hot_true, probabilities, average="macro")
        )
    except ValueError:
        aggregate["average_precision_macro"] = None

    per_class = pd.DataFrame({
        "class_id": range(len(CLASS_ORDER)),
        "class": CLASS_ORDER,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support.astype(int),
    })
    matrix = confusion_matrix(y_true, predictions, labels=list(range(len(CLASS_ORDER))))
    return aggregate, per_class, matrix, predictions, probabilities

def compute_trainer_metrics(eval_prediction):
    aggregate, _, _, _, _ = metric_bundle(eval_prediction.label_ids, eval_prediction.predictions)
    return aggregate

In [ ]:
# Use weighted cross-entropy with training-only class weights and version-compatible Trainer APIs.
class WeightedCrossEntropyTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is None:
            raise ValueError("class_weights are required")
        self.class_weights = class_weights
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = F.cross_entropy(
            outputs.logits,
            labels,
            weight=self.class_weights.to(outputs.logits.device),
        )
        return (loss, outputs) if return_outputs else loss

FP16_ENABLED = bool(
    CUDA_AVAILABLE and CUDA_CAPABILITY is not None and CUDA_CAPABILITY[0] >= 7
)
actual_batch_size = PER_DEVICE_BATCH_SIZE if CUDA_AVAILABLE else 2
strategy_argument = (
    "eval_strategy"
    if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters
    else "evaluation_strategy"
)
training_argument_values = {
    "output_dir": str(CHECKPOINT_DIR),
    "learning_rate": LEARNING_RATE,
    "num_train_epochs": 1 if SMOKE_TEST else EPOCHS,
    "per_device_train_batch_size": actual_batch_size,
    "per_device_eval_batch_size": actual_batch_size,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    strategy_argument: "epoch",
    "save_strategy": "no" if SMOKE_TEST else "epoch",
    "logging_strategy": "epoch",
    "load_best_model_at_end": not SMOKE_TEST,
    "metric_for_best_model": "macro_f1" if not SMOKE_TEST else None,
    "greater_is_better": True if not SMOKE_TEST else None,
    "save_total_limit": 2,
    "fp16": FP16_ENABLED,
    "seed": SEED,
    "data_seed": SEED,
    "report_to": [],
    "push_to_hub": False,
    "auto_find_batch_size": True,
    "dataloader_num_workers": 2 if CUDA_AVAILABLE else 0,
    "max_steps": SMOKE_MAX_STEPS if SMOKE_TEST else -1,
}
training_args = TrainingArguments(**training_argument_values)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_dataset["train"],
    "eval_dataset": tokenized_dataset["dev"],
    "data_collator": data_collator,
    "compute_metrics": compute_trainer_metrics,
    "class_weights": class_weights_tensor,
    "callbacks": [] if SMOKE_TEST else [
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)
    ],
}
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = WeightedCrossEntropyTrainer(**trainer_kwargs)
print("Initial per-device batch size:", actual_batch_size)
print("Automatic OOM batch-size reduction enabled:", training_args.auto_find_batch_size)
print("FP16 enabled:", FP16_ENABLED)

In [ ]:
# Train on train only, select using dev only, and recover development results from the selected state.
training_started = time.perf_counter()
train_result = trainer.train()
training_time_seconds = float(time.perf_counter() - training_started)

dev_started = time.perf_counter()
dev_prediction = trainer.predict(tokenized_dataset["dev"], metric_key_prefix="dev_final")
dev_inference_time_seconds = float(time.perf_counter() - dev_started)
dev_metrics, dev_per_class_df, dev_confusion_matrix, _, _ = metric_bundle(
    dev_prediction.label_ids, dev_prediction.predictions
)

dev_history = [
    record for record in trainer.state.log_history
    if "eval_macro_f1" in record and record.get("epoch") is not None
]
best_dev_epoch = (
    float(max(dev_history, key=lambda record: record["eval_macro_f1"])["epoch"])
    if dev_history else float(trainer.state.epoch or 0.0)
)
effective_train_batch_size = int(getattr(trainer, "_train_batch_size", actual_batch_size))

print("Training time (seconds):", training_time_seconds)
print("Best development epoch:", best_dev_epoch)
print("Development metrics:")
print(json.dumps(dev_metrics, indent=2))
display(dev_per_class_df)

In [ ]:
# Save a clearly labeled smoke-validation record, or evaluate the untouched test split exactly once.
test_metrics = None
test_per_class_df = None
test_confusion_matrix = None
test_inference_time_seconds = None
test_evaluation_calls = 0

if SMOKE_TEST:
    smoke_payload = {
        "status": "smoke_test_passed",
        "invalid_for_paper": True,
        "test_evaluated": False,
        "model_checkpoint": MODEL_CHECKPOINT,
        "seed": SEED,
        "small_batch_logits_shape": list(probe_logits.shape),
        "smoke_train_rows": int(len(train_frame)),
        "smoke_dev_rows": int(len(dev_frame)),
        "training_time_seconds": training_time_seconds,
        "best_development_epoch": best_dev_epoch,
        "smoke_development_metrics": dev_metrics,
        "hardware": hardware,
    }
    with open(OUTPUT_DIR / "smoke_test_validation.json", "w", encoding="utf-8") as file:
        json.dump(smoke_payload, file, ensure_ascii=False, indent=2)
    display(Markdown(
        "**Smoke validation passed. These sampled development metrics are invalid for the paper. "
        "The test split was not evaluated and no final comparison files were produced.**"
    ))
else:
    assert CUDA_AVAILABLE, "The full experiment requires a CUDA-enabled Colab runtime."
    assert len(tokenized_dataset["test"]) == EXPECTED_ROWS["test"]
    test_started = time.perf_counter()
    test_prediction = trainer.predict(tokenized_dataset["test"], metric_key_prefix="test")
    test_evaluation_calls += 1
    test_inference_time_seconds = float(time.perf_counter() - test_started)
    test_metrics, test_per_class_df, test_confusion_matrix, test_predictions, test_probabilities = metric_bundle(
        test_prediction.label_ids, test_prediction.predictions
    )
    print("Untouched test metrics:")
    print(json.dumps(test_metrics, indent=2))
    display(test_per_class_df)

In [ ]:
# In a full GPU run only, persist actual six-class results, figures, comparison data, and ignored model files.
if not SMOKE_TEST:
    assert test_evaluation_calls == 1, "The test split must be evaluated exactly once."
    assert len(test_prediction.label_ids) == EXPECTED_ROWS["test"]
    assert set(np.unique(test_prediction.label_ids)) == set(range(len(CLASS_ORDER)))
    assert int(test_confusion_matrix.sum()) == EXPECTED_ROWS["test"]
    assert int(test_per_class_df["support"].sum()) == EXPECTED_ROWS["test"]
    assert test_per_class_df["class"].tolist() == CLASS_ORDER
    assert all(np.isfinite(value) for value in test_metrics.values() if value is not None)

    package_versions = {}
    for package_name in [
        "torch", "transformers", "datasets", "accelerate",
        "pandas", "numpy", "scikit-learn", "matplotlib", "seaborn"
    ]:
        try:
            package_versions[package_name] = importlib_metadata.version(package_name)
        except importlib_metadata.PackageNotFoundError:
            package_versions[package_name] = None

    final_metrics_payload = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "paper_valid_full_run": True,
        "test_evaluated_exactly_once": True,
        "model_checkpoint": MODEL_CHECKPOINT,
        "class_order": CLASS_ORDER,
        "label_to_id": LABEL_TO_ID,
        "seed": SEED,
        "hyperparameters": {
            "max_sequence_length": MAX_LENGTH,
            "learning_rate": LEARNING_RATE,
            "epochs_requested": EPOCHS,
            "initial_per_device_batch_size": PER_DEVICE_BATCH_SIZE,
            "effective_per_device_batch_size": effective_train_batch_size,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "fp16": FP16_ENABLED,
            "auto_find_batch_size": True,
        },
        "split_sizes": EXPECTED_ROWS,
        "class_distributions": class_distribution_df.to_dict(orient="records"),
        "training_class_weights": class_weight_payload,
        "hardware": hardware,
        "package_versions": package_versions,
        "parameter_count": PARAMETER_COUNT,
        "trainable_parameter_count": TRAINABLE_PARAMETER_COUNT,
        "best_development_epoch": best_dev_epoch,
        "best_model_checkpoint": trainer.state.best_model_checkpoint,
        "development_metrics": dev_metrics,
        "test_metrics": test_metrics,
        "training_time_seconds": training_time_seconds,
        "development_inference_time_seconds": dev_inference_time_seconds,
        "test_inference_time_seconds": test_inference_time_seconds,
    }
    with open(OUTPUT_DIR / "banglabert_metrics.json", "w", encoding="utf-8") as file:
        json.dump(final_metrics_payload, file, ensure_ascii=False, indent=2)

    test_per_class_df.to_csv(OUTPUT_DIR / "banglabert_per_class_metrics.csv", index=False)
    confusion_df = pd.DataFrame(test_confusion_matrix, index=CLASS_ORDER, columns=CLASS_ORDER)
    confusion_df.index.name = "true_label"
    confusion_df.to_csv(OUTPUT_DIR / "banglabert_confusion_matrix.csv")

    comparison_df = pd.DataFrame([
        {
            "model": classical_reference["model"],
            "model_family": "Classical linear classifier",
            "imbalance_method": "Balanced class weighting",
            "development_macro_f1": classical_reference["development_macro_f1"],
            "test_accuracy": classical_reference["test_accuracy"],
            "test_micro_f1": classical_reference["test_micro_f1"],
            "test_macro_f1": classical_reference["test_macro_f1"],
            "test_weighted_f1": classical_reference["test_weighted_f1"],
            "test_balanced_accuracy": classical_reference["test_balanced_accuracy"],
            "minority_recall_mean": classical_reference["minority_recall_mean"],
            "training_time_seconds": np.nan,
            "test_inference_time_seconds": np.nan,
            "parameter_count": np.nan,
        },
        {
            "model": "BanglaBERT",
            "model_family": "Transformer encoder",
            "imbalance_method": "Training-only weighted cross-entropy",
            "development_macro_f1": dev_metrics["macro_f1"],
            "test_accuracy": test_metrics["accuracy"],
            "test_micro_f1": test_metrics["micro_f1"],
            "test_macro_f1": test_metrics["macro_f1"],
            "test_weighted_f1": test_metrics["weighted_f1"],
            "test_balanced_accuracy": test_metrics["balanced_accuracy"],
            "minority_recall_mean": test_metrics["minority_recall_mean"],
            "training_time_seconds": training_time_seconds,
            "test_inference_time_seconds": test_inference_time_seconds,
            "parameter_count": PARAMETER_COUNT,
        },
    ])
    comparison_df.to_csv(OUTPUT_DIR / "classical_vs_banglabert.csv", index=False)
    with open(OUTPUT_DIR / "classical_vs_banglabert.tex", "w", encoding="utf-8") as file:
        file.write(comparison_df.to_latex(index=False, float_format="%.4f"))

    sns.set_theme(style="whitegrid", context="paper")
    plt.figure(figsize=(8.5, 7.2))
    sns.heatmap(confusion_df, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("BanglaBERT Test Confusion Matrix")
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "banglabert_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()

    comparison_plot = comparison_df.melt(
        id_vars="model",
        value_vars=[
            "test_accuracy", "test_micro_f1", "test_macro_f1",
            "test_weighted_f1", "test_balanced_accuracy", "minority_recall_mean"
        ],
        var_name="metric",
        value_name="score",
    )
    comparison_plot["metric"] = comparison_plot["metric"].str.replace("test_", "", regex=False).str.replace("_", " ")
    plt.figure(figsize=(10.5, 5.8))
    sns.barplot(data=comparison_plot, x="metric", y="score", hue="model")
    plt.ylim(0, 1)
    plt.title("Corrected Classical Model vs BanglaBERT on the Untouched Test Split")
    plt.xlabel("")
    plt.ylabel("Score")
    plt.xticks(rotation=25, ha="right")
    plt.legend(title="Model", frameon=True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "classical_vs_banglabert.png", dpi=300, bbox_inches="tight")
    plt.close()

    final_model_dir = CHECKPOINT_DIR / "final_model"
    trainer.save_model(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)

    required_outputs = [
        "banglabert_metrics.json",
        "banglabert_per_class_metrics.csv",
        "banglabert_confusion_matrix.csv",
        "classical_vs_banglabert.csv",
        "banglabert_confusion_matrix.png",
        "classical_vs_banglabert.png",
        "classical_vs_banglabert.tex",
    ]
    for filename in required_outputs:
        assert (OUTPUT_DIR / filename).exists(), f"Missing required output: {filename}"

    print("Paper-valid full six-class BanglaBERT outputs saved:")
    for filename in required_outputs:
        print("-", OUTPUT_DIR / filename)

## Interpretation guardrails

- The training split alone determines class weights and model parameters.
- Development macro-F1 selects the best epoch; test data is never used for selection.
- The untouched test split is evaluated exactly once and only in a full GPU run.
- Missing classical timing and parameter-count values remain missing because Phase 4 did not record them.
- Smoke-mode metrics are explicitly invalid for the paper and never populate the final comparison files.
- This experiment intentionally contains no SHAP or LIME analysis.